# PubMed embeddings on one free Colab runtime

This notebook creates embeddings; it does not fine-tune a language model. Completed parts are saved to Google Drive and resume safely after a disconnect. Read `colab/README.md` before running.

In [ ]:
# Free Colab default: one resumable worker.
USE_GIT = False  # False: upload the small code ZIP; True: clone a pushed branch.
REPO_URL = 'https://github.com/nischalpatil82/pubmed-ai.git'
BRANCH = 'covid-files'
DATASET_MANIFEST = '/content/drive/MyDrive/pubmed-ai-data/dataset.json'
TOTAL_WORKERS = 1
ROWS_PER_PART = 10_000
GPU_BATCH = 128
MAX_RUNTIME_MINUTES = 270  # Use 4.5 hours, then stop cleanly so checkpoints finish saving.
MODEL = 'BAAI/bge-small-en-v1.5'
ALLOW_PILOT = False  # Change to True only for a deliberate small test.
UPLOAD_PILOT_ARCHIVE = False  # True uploads the supplied 28 MB pilot package for benchmarking.
WORKER_ID = 0
assert 0 <= WORKER_ID < TOTAL_WORKERS

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, subprocess, sys
if UPLOAD_PILOT_ARCHIVE:
    from google.colab import files
    import zipfile
    uploaded = files.upload()
    pilot = next((name for name in uploaded if name == 'pubmed-pilot-articles.zip'), None)
    assert pilot, 'Upload pubmed-pilot-articles.zip'
    os.makedirs(os.path.dirname(DATASET_MANIFEST), exist_ok=True)
    with zipfile.ZipFile(pilot) as archive:
        archive.extractall(os.path.dirname(DATASET_MANIFEST))
assert shutil.which('nvidia-smi'), 'Select Runtime > Change runtime type > T4 GPU'
subprocess.run(['nvidia-smi'], check=True)
usage = shutil.disk_usage('/content')
print(f'Runtime disk free: {usage.free / 1e9:.1f} GB')

In [ ]:
# Load the exact code. Direct upload works before the branch is pushed.
PROJECT = '/content/pubmed-ai'
if USE_GIT:
    if not os.path.exists(os.path.join(PROJECT, '.git')):
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, PROJECT], check=True)
    else:
        subprocess.run(['git', '-C', PROJECT, 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    from google.colab import files
    import zipfile
    uploaded = files.upload()
    package = next((name for name in uploaded if name.endswith('.zip')), None)
    assert package, 'Upload pubmed-ai-colab-code.zip'
    os.makedirs(PROJECT, exist_ok=True)
    with zipfile.ZipFile(package) as archive:
        archive.extractall(PROJECT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', os.path.join(PROJECT, 'requirements.txt')], check=True)

In [ ]:
# Validate that this runtime points at a complete, immutable dataset.
import json
from pathlib import Path
manifest_path = Path(DATASET_MANIFEST)
assert manifest_path.exists(), f'Missing {manifest_path}'
dataset = json.loads(manifest_path.read_text())
assert dataset.get('status') == 'ready'
if dataset.get('pilot') and not ALLOW_PILOT:
    raise RuntimeError('This manifest is a pilot. Build the full 159-member store, or set ALLOW_PILOT=True only for a test.')
store = (manifest_path.parent / dataset['store']).resolve()
articles = store / 'articles.parquet'
assert articles.exists(), f'Missing {articles}'
print('Snapshot:', dataset['snapshot'])
print('Articles file:', articles)
os.environ['PUBMED_DATASET'] = str(manifest_path.resolve())
os.environ['PUBMED_EMBED_DEVICE'] = 'cuda'
os.environ['PUBMED_ALLOW_MODEL_DOWNLOAD'] = '1'

## Benchmark first
Run this before starting the worker. The result measures the GPU assigned to this session and estimates one-GPU time from 10,000 articles.

In [ ]:
command = [sys.executable, os.path.join(PROJECT, 'pipeline', 'embedding_shards.py'), 'benchmark',
           '--dataset', DATASET_MANIFEST, '--model', MODEL,
           '--benchmark-articles', '10000', '--gpu-batch', str(GPU_BATCH)]
subprocess.run(command, check=True)

## Run this account's worker
Rerun this cell after a disconnection. Finished parts are skipped. If GPU memory is exhausted, lower GPU_BATCH; existing parts remain valid.

In [ ]:
command = [sys.executable, os.path.join(PROJECT, 'pipeline', 'embedding_shards.py'), 'worker',
           '--dataset', DATASET_MANIFEST, '--model', MODEL, '--worker', str(WORKER_ID),
           '--workers', str(TOTAL_WORKERS), '--rows-per-part', str(ROWS_PER_PART),
           '--gpu-batch', str(GPU_BATCH), '--max-runtime-minutes', str(MAX_RUNTIME_MINUTES)]
subprocess.run(command, check=True)

## Verify after the worker finishes
Run from either account. It checks every expected part and checksum. Do not finalize until this passes.

In [ ]:
command = [sys.executable, os.path.join(PROJECT, 'pipeline', 'embedding_shards.py'), 'verify',
           '--dataset', DATASET_MANIFEST, '--workers', str(TOTAL_WORKERS),
           '--rows-per-part', str(ROWS_PER_PART)]
subprocess.run(command, check=True)

## Finalize once
Run this in only one account. It creates the final Lance vector index and keeps source passages in compressed Parquet. It may take significant time and temporarily needs space for both vector parts and the final index.

In [ ]:
assert WORKER_ID == 0, 'The single worker must be worker 0'
command = [sys.executable, os.path.join(PROJECT, 'pipeline', 'embedding_shards.py'), 'finalize',
           '--dataset', DATASET_MANIFEST, '--workers', str(TOTAL_WORKERS),
           '--rows-per-part', str(ROWS_PER_PART)]
subprocess.run(command, check=True)